[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/second_harmonic/second_harmonic.ipynb)

# Second-Harmonic Generation

This example computes second-harmonic generation in a tetrahedral crystal whose electric response is quadratic in the driving field, converting infrared pump light into light at twice the frequency. Conversion strength depends on how the crystal bonds align against the pump polarization across the beam, and coherent buildup through a thick crystal depends on whether successive slices stay in phase or flip their bond orientation whenever dispersion slips them half a cycle apart.

In spacetime, the electric field is modelled as an extrinsic bivector, while spatial directions and the observer's time direction are intrinsic antivectors whose join `bond_directions & time` is the intrinsic antibivector plane each bond sweeps in time. Vacuum duality `(Direction & time).dual()` maps each spatial direction to the electric field along it, pairing `Field & bonds` reads each bond's share of a field, and summing `bonds * (Field & bonds) * (Field & bonds)` builds the bulk bilinear response so `rotation >> crystal(rotation << Field, rotation << Field)` rotates it directly with the crystal. The tetrahedral bond model follows [Hardhienata et al., *Bond Model and Group Theory of Second Harmonic Generation in GaAs(001)*](https://arxiv.org/abs/1408.1185), and the weak-conversion phase accumulation follows [MIT's *Optical Properties of Solids*, chapter 11](https://web.mit.edu/6.732/www/opt.pdf).

In Cartesian nonlinear-optics tensor notation, the second-order electric-dipole polarization $P_i(2\omega) = \varepsilon_0 \sum_{j,k} \chi^{(2)}_{ijk} E_j(\omega) E_k(\omega)$ contracts a 27-component third-rank susceptibility tensor (often compressed into a $3 \times 6$ Voigt matrix $d_{il}$ with factor-of-two off-diagonal conventions), and rotating the crystal transforms those components via the triple matrix product $\chi'^{(2)}_{ijk} = \sum_{a,b,c} R_{ia} R_{jb} R_{kc} \chi^{(2)}_{abc}$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA
from examples.util.animation import save_figure, show_states
from examples.electromagnetism.second_harmonic import render, scenarios

ga = STA
mv = NumpyContext(ga).multivector
# Fields are extrinsic bivectors; directions are intrinsic antivectors; polarizations are the
# intrinsic antibivector planes that spatial directions sweep in time.
Field = ga.gatype.bivector()                            # Bivector+
Direction = ga.gatype.antivector()                      # Antivector-
Polarization = ga.gatype.antibivector()                 # Antibivector-
Response = ga.gatype((Polarization, Field, Field))      # Antibivector- <- (Bivector+, Bivector+)

# Vacuum duality identifies each spatial direction with the electric field dual to the plane it sweeps in time.
time = -mv.t.dual()                                     # Direction-
field_along = (Direction & time).dual()                 # [] Field+ <- Direction-
# Transverse directions across a beam along x + y, together with their polarizations and unit fields.
axes = stack((mv.x.dual(), mv.y.dual(), mv.z.dual()))   # [3] Direction-
horizontal_step = mv.y.dual() - mv.x.dual()             # Direction-
horizontal = horizontal_step / horizontal_step.antinorm()  # Direction-
vertical = mv.z.dual()                                  # Direction-
horizontal_polarization = horizontal & time             # Polarization-
vertical_polarization = vertical & time                 # Polarization-
horizontal_field = field_along(horizontal)              # Field+
vertical_field = field_along(vertical)                  # Field+
# Transverse projection onto the two polarizations carried across the beam.
transverse = (stack((horizontal_polarization, vertical_polarization))
              * (stack((horizontal_field, vertical_field)) & Polarization)).sum(axis=0)   # [] Polarization- <- Polarization-
# Bivector plane generating spatial rotations about the beam axis.
turn_plane = horizontal_field.commutator(vertical_field)                                  # Bivector+
# Four tetrahedral bond directions, their spacetime planes, and the bond strength giving unit bulk coefficients.
bond_steps = (axes * np.array([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]])).sum(axis=-1)   # [4] Direction-
bond_directions = bond_steps / bond_steps.antinorm()                                      # [4] Direction-
bonds = bond_directions & time                          # [4] Polarization-
bond_strength = 3 * np.sqrt(3) / 4

## Quadratic bond response

Each tetrahedral bond responds along its own spacetime plane in proportion to the product of the two input fields' pairings with that plane, and summing over the four bonds yields the bulk crystal response. A sinusoidal pump field drives that quadratic map as a squared cosine, splitting equally between a static optical-rectification offset and a polarization oscillating at twice the pump frequency.

In Cartesian tensor notation, a zincblende crystal in its cubic axes has $\chi^{(2)}_{xyz} = \chi^{(2)}_{yzx} = \chi^{(2)}_{zxy}$ as its sole independent components, and $\cos^2(\omega t) = \tfrac12(1 + \cos 2\omega t)$ separates optical rectification from the $2\omega$ polarization.

In [ ]:
n_periods = scenarios.n_periods
n_waveform_samples = scenarios.n_waveform_samples

In [ ]:
def bond_response(bonds: Polarization) -> Response:
    """Sums each bond's quadratic response along its plane (`[4] Polarization-`) into a bulk response
    map (`[] Polarization- <- (Field+, Field+)`)."""
    return bond_strength * (bonds * (Field & bonds) * (Field & bonds)).sum(axis=-1)


crystal = bond_response(bonds)                                                        # [] Polarization- <- (Field+, Field+)
# Cosine-squared oscillation splits equally between a static offset and the doubled-frequency component.
second_harmonic = 0.5 * transverse(crystal)                                           # [] Polarization- <- (Field+, Field+)

phase = np.linspace(0, 2 * np.pi * n_periods, n_waveform_samples)
pump = horizontal_field * np.cos(phase)                                               # [n_waveform_samples] Field+
# Subtracting the static half leaves the pure doubled-frequency oscillation.
harmonic = (transverse(crystal(pump, pump))
            - second_harmonic(horizontal_field, horizontal_field))                    # [n_waveform_samples] Polarization-

save_figure(render.draw_waveform(phase, pump, harmonic), "second_harmonic_waveform");

The blue curve traces the horizontal pump field over two optical cycles, while the orange curve shows the transverse vertical polarization oscillating through four full cycles after subtracting the static offset.

## Crystal rotation about the beam

Rotating the crystal rotates its bilinear response: `rotation >> second_harmonic(rotation << pumps, rotation << pumps)` pulls every pump field into the crystal frame, evaluates the transverse doubled polarization there, and rotates the polarization back to the lab screen. Sweeping the pump heading around the beam maps out the full polarization figure and directional power lobes, while turning the crystal about the beam rotates that pattern across a fixed horizontal pump.

In [ ]:
n_headings = 361
n_turns = scenarios.n_turns
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
headings = np.linspace(0, 2 * np.pi, n_headings)
heading_turns = (turn_plane * (-headings / 2)).exp()                                  # [n_headings] Rotor+
pumps = heading_turns >> horizontal_field                                             # [n_headings] Field+
save_figure(render.draw_response(pumps, bond_directions, second_harmonic(pumps, pumps)), "second_harmonic_response");

In [ ]:
turns = np.linspace(0, 2 * np.pi, n_turns, endpoint=False)
rotations = (turn_plane * (-turns / 2)).exp()                                         # [n_turns] Rotor+


def turning_states() -> Iterator[tuple[Direction, Polarization]]:
    """Yields the rotated bond directions (`[4] Direction-`) and transverse doubled polarizations
    (`[n_headings] Polarization-`) across `n_turns` crystal rotations about the beam."""
    for rotation in rotations:
        back_pumps = rotation << pumps                                                # [n_headings] Field+
        yield rotation >> bond_directions, rotation >> second_harmonic(back_pumps, back_pumps)


kept_states = show_states(turning_states(), render.animate, pumps,
                          name="second_harmonic", duration_ms=frame_duration_ms, n_frames=n_turns)

From left to right, the three panels show the four tetrahedral bonds around the incoming beam, the orange doubled-polarization locus driven by the blue circle of unit pumps (with arrows marking the horizontal pump and its response), and the polar intensity lobes with a blue dot tracking the horizontal pump's output power as the crystal turns.

## Linear mixing of weak probes on a fixed pump

Binding a strong pump field into one argument of `crystal` leaves a linear map `transverse(crystal(pump_fields[:, None], Field))` on weak probe fields. Because the bond sum is symmetric in its two field inputs, binding either input gives the same linear transformation, mapping a circle of weak transverse probes into an ellipse or segment depending on the fixed pump's direction.

In Fréchet-derivative notation, expanding $P(E + \delta E) - P(E) - P(\delta E) = 2\varepsilon_0 \chi^{(2)}(E, \delta E)$ isolates the symmetric bilinear cross term responsible for sum-frequency mixing.

In [ ]:
pump_fields = stack((horizontal_field, (horizontal_field + vertical_field) / np.sqrt(2)))  # [2] Field+
probe_amplitude = scenarios.probe_amplitude

In [ ]:
probes = probe_amplitude * pumps                                                      # [n_headings] Field+
# Binding a fixed pump into one input leaves a linear map on probe fields.
probe_map = transverse(crystal(pump_fields[:, None], Field))                          # [2, 1] Polarization- <- Field+
generated = probe_map(probes)                                                         # [2, n_headings] Polarization-

save_figure(render.draw_mixing(pump_fields, probes, generated), "second_harmonic_mixing");

A pale grey arrow marks the fixed pump direction in each panel, the dashed blue circle traces the weak probe headings, and the solid orange curve shows the mixed polarization image: an ellipse for the horizontal pump on the left and a tilted ellipse for the diagonal pump on the right.

## Coherent growth and quasi-phase matching

Successive crystal slices radiate doubled light that adds coherently only while the fundamental pump and second harmonic travel in step. Dispersion introduces a linear phase slip with depth that rotates the cosine and sine quadratures of each slice's contribution around a circle so uniform mismatched slices periodically cancel. Inverting the tetrahedral bonds every half turn of phase slip flips the sign of `crystal` right where destructive interference would start, restoring monotonic growth across a periodically poled crystal.

In complex slowly-varying-envelope notation, weak-conversion phase accumulation integrates $A_{2\omega}(L) = \int_0^L s(z)\,e^{i\Delta k\,z}\,dz$ with domain polarity $s(z) = \pm 1$ and wavevector mismatch $\Delta k = k_{2\omega} - 2 k_\omega$.

In [ ]:
n_slices = scenarios.n_slices
slip = scenarios.slip

In [ ]:
def growth(source: Polarization, bond_orientation: np.ndarray, mismatch: np.ndarray,
           depths: np.ndarray) -> Polarization:
    """Accumulates the cosine and sine quadratures of `source` across slices (`[..., 2] Polarization-`),
    weighted by each slice's `bond_orientation` and phase slip `mismatch * depths`."""
    thickness = depths[1] - depths[0]
    phase_slip = mismatch * depths
    quadratures = stack((source[..., None] * np.cos(phase_slip),
                         source[..., None] * np.sin(phase_slip)), axis=-1)            # [..., 2] Polarization-
    return (quadratures * bond_orientation[..., None] * thickness).cumsum(axis=-2)


depths = (np.arange(n_slices) + 0.5) / n_slices
mismatch = np.array([[0], [slip], [slip]])
# Inverting the tetrahedral bonds reverses the quadratic response every half turn of phase slip.
flips = (-1.0) ** np.floor(depths * slip / np.pi)
bond_orientation = np.stack([np.ones(n_slices), np.ones(n_slices), flips])
amplitude = growth(vertical_polarization, bond_orientation, mismatch, depths)         # [3, n_slices, 2] Polarization-

save_figure(render.draw_growth(depths, amplitude), "second_harmonic_phase_matching");

In the phase plane on the left and the power plot on the right, the blue phase-matched crystal marches straight along the in-phase axis with quadratic power growth, the orange mismatched crystal loops back to the origin every full turn of slip, and the purple periodically poled crystal chains semicircles into steady net growth.

In [ ]:
# checks
# Unrotated horizontal pump drives a negative half-unit vertical polarization; a quarter turn extinguishes it
# and a half turn reverses its sign.
_, first_harmonic = kept_states[0]
_, quarter_turn_harmonic = kept_states[n_turns // 4]
_, half_turn_harmonic = kept_states[n_turns // 2]
assert ((first_harmonic[0] + vertical_polarization / 2).norm_squared() < 1e-12).all()
assert (quarter_turn_harmonic[0].norm_squared() < 1e-12).all()
assert ((half_turn_harmonic[0] - vertical_polarization / 2).norm_squared() < 1e-12).all()
# Pump-bound linear response equals the cross term of pump plus probe.
pump_and_probe = pump_fields[:, None] + probes
mixed = (second_harmonic(pump_and_probe, pump_and_probe)
         - second_harmonic(pump_fields[:, None], pump_fields[:, None])
         - second_harmonic(probes, probes))
assert ((mixed - generated).norm_squared() < 1e-12).all()
# Matched power reaches 1, mismatched cancels at whole-turn slip, and quasi-phase-matched reaches (2 / pi)^2.
power = amplitude.scalar_norm_squared().sum(axis=-1)
assert ((power[:, -1] - np.array([1.0, 0.0, (2 / np.pi) ** 2])).abs() < 1e-4).all()